# Vision AI Studio — Phase 5: BLIP LoRA Fine-Tuning

This notebook fine-tunes the **BLIP** (Salesforce/blip-image-captioning-base) captioning model using **LoRA (Low-Rank Adaptation)** on GPU.

### Why LoRA?
- Full fine-tuning of BLIP requires updating all ~223M parameters.
- With LoRA ($r=16, \alpha=32$), we freeze the vision backbone and only inject low-rank decomposition matrices into the text decoder attention/projection layers.
- **<1.5% trainable parameters**, fits comfortably on free Colab/Kaggle T4 GPUs, and outputs a lightweight adapter (~15 MB) instead of a multi-gigabyte checkpoint.

In [ ]:
# 1. Verify GPU availability
!nvidia-smi

In [ ]:
# 2. Clone repo and install dependencies
!git clone https://github.com/EBHAVANI17/VISION_AI_STUDIO.git
%cd VISION_AI_STUDIO
!pip install -q peft transformers accelerate pycocotools mlflow
!pip install -q -e .

In [ ]:
# 3. Run LoRA training
from vision_ai_studio.training.blip_lora import BlipLoraTrainingConfig, train_blip_lora
from vision_ai_studio.tracking import get_tracker
from PIL import Image

config = BlipLoraTrainingConfig(
    base_model='Salesforce/blip-image-captioning-base',
    r=16,
    alpha=32,
    epochs=2,
    batch_size=8,
    lr=1e-4,
    device='cuda',
    output_dir='artifacts/models/blip-lora'
)

# Sample training invocation
print('Configured LoRA fine-tuning for BLIP:', config)

In [ ]:
# 4. Launch training CLI
!python scripts/train_blip_lora.py --epochs 2 --batch-size 8 --tracking none

### 5. Export Adapter to Hugging Face Hub (Optional)
Once training finishes, upload the lightweight LoRA adapter directly to your Hugging Face account so Vision AI Studio can load it via `LORA_REPO`!

In [ ]:
# from huggingface_hub import login
# login()
# from peft import PeftModel
# from transformers import BlipForConditionalGeneration
# base = BlipForConditionalGeneration.from_pretrained('Salesforce/blip-image-captioning-base')
# model = PeftModel.from_pretrained(base, 'artifacts/models/blip-lora')
# model.push_to_hub('EBHAVANI17/vision-ai-blip-coco-lora')